# Prueba de conexión Jupyter ↔ HDFS
Comprueba que este Jupyter puede **leer** los CSV de `/datalake/raw` y **escribir** Parquet en HDFS (WebHDFS). Ejecuta las celdas en orden.

In [1]:
import os
import pandas as pd

# Dirección de HDFS por WebHDFS (viene de las variables de entorno del contenedor)
HOST = os.environ.get('HDFS_WEBHDFS_URL', 'http://namenode:9870').replace('http://', '')
OPT = {'user': os.environ.get('HDFS_USER', 'root')}   # usuario con el que se accede a HDFS
RAW = f'webhdfs://{HOST}/datalake/raw'
CLEAN = f'webhdfs://{HOST}/datalake/clean'
print(RAW)

webhdfs://namenode:9870/datalake/raw


## 1. Leer un CSV desde HDFS

In [2]:
# Leemos solo 5 filas del CSV en bruto. dtype=str porque los datos están sucios.
df = pd.read_csv(f'{RAW}/historico/afluencia_diaria_historico.csv', storage_options=OPT, nrows=5, dtype=str)
df

,id_registro,id_restaurante,fecha_servicio,ciudad,provincia,tipo_restaurante,capacidad_asientos,dia_semana,mes,temporada,festivo_simulado,evento_local_previsto,temperatura_prevista_c,lluvia_prevista_mm,horas_apertura_planificadas,fecha_corte_prediccion,clientes_reales,clientes_con_reserva_reales,clientes_sin_reserva_reales
0,AFL-RST-0001-20240101,RST-0001,2024-01-01,Donostia,Gipuzkoa,casual,58,lunes,1,invierno,1,1,7.6,0,8,2023-12-31T23:59:00,44,31,13
1,AFL-RST-0002-20240101,RST-0002,2024-01-01,Bilbao,Bizkaia,bistró,49,lunes,1,invierno,1,0,6.2,0,8,2023-12-31T23:59:00,31,19,12
2,AFL-RST-0003-20240101,RST-0003,2024-01-01,Vitoria-Gasteiz,Araba,familiar,74,lunes,1,invierno,1,0,8.5,0,8,2023-12-31T23:59:00,57,39,18
3,AFL-RST-0004-20240101,RST-0004,2024-01-01,Pamplona,Navarra,tradicional,99,lunes,1,invierno,1,0,7.7,0,8,2023-12-31T23:59:00,66,40,26
4,AFL-RST-0005-20240101,RST-0005,2024-01-01,Santander,Cantabria,casual,95,lunes,1,invierno,1,0,8.8,1.7,8,2023-12-31T23:59:00,62,41,21


## 2. Escribir un Parquet en HDFS

In [3]:
# Escribimos esas 5 filas en una carpeta de prueba dentro de /datalake/clean
df.to_parquet(f'{CLEAN}/_prueba/test.parquet', storage_options=OPT, index=False)
print('Parquet escrito')

Parquet escrito


## 3. Leerlo de vuelta y comprobar

In [5]:
df2 = pd.read_parquet(f'{CLEAN}/_prueba/test.parquet', storage_options=OPT)
print(df2.shape, '- iguales:', df.equals(df2))
df2.dtypes

(5, 19) - iguales: True


id_registro                    object
id_restaurante                 object
fecha_servicio                 object
ciudad                         object
provincia                      object
tipo_restaurante               object
capacidad_asientos             object
dia_semana                     object
mes                            object
temporada                      object
festivo_simulado               object
evento_local_previsto          object
temperatura_prevista_c         object
lluvia_prevista_mm             object
horas_apertura_planificadas    object
fecha_corte_prediccion         object
clientes_reales                object
clientes_con_reserva_reales    object
clientes_sin_reserva_reales    object
dtype: object

## 4. Borrar el archivo de prueba

In [6]:
import fsspec
fs = fsspec.filesystem('webhdfs', host=HOST.split(':')[0], port=int(HOST.split(':')[1]), **OPT)
fs.rm(f'/datalake/clean/_prueba', recursive=True)
print('Prueba borrada:', not fs.exists('/datalake/clean/_prueba'))

Prueba borrada: True
